# Fabric Data Agent evaluation notebook

This notebook implements the evaluation workflow described in the article.  
It starts from the benchmark Excel file with `question` and `expected_answer`, runs the official Fabric evaluation flow, inspects both summary and row-level results, exports an audit file, and finally reruns the benchmark with a stricter custom critic prompt to reduce evaluator noise.


## 1. Install the Fabric Data Agent SDK

The Fabric Python notebook environment already includes many common libraries, but `fabric-data-agent-sdk` is not preinstalled by default.

We install the latest available SDK version directly in the notebook kernel so that the evaluation functions such as `evaluate_data_agent`, `get_evaluation_summary`, and `get_evaluation_details` become available.


In [ ]:
%pip install -U fabric-data-agent-sdk

## 2. Load the completed benchmark from OneLake

At this stage, the benchmark Excel file already contains the `expected_answer` column populated from the canonical questions and propagated to the three corresponding variants.

The file is stored in the supporting Lakehouse and accessed through its `abfss://...` OneLake path.  
We load the full Excel into a pandas DataFrame so that we can use the minimal evaluation view for execution while still preserving all benchmark metadata for later analysis.


In [ ]:
import pandas as pd

df = pd.read_excel("abfss://<workspace_guid>@onelake.dfs.fabric.microsoft.com/<lakehouse_guid>/Files/final_benchmark_with_expected_answers.xlsx")

display(df)

## 3. Configure the evaluation run

Here we define the basic runtime configuration:

- the name of the target Data Agent
- the optional workspace override
- the name of the output table that will store evaluation results
- the target stage of the Data Agent (`sandbox` or `production`)

The output table name is important because Fabric will persist both the main evaluation results and the related step-level execution data in the notebook's default Lakehouse.


In [ ]:
# Name of your Data Agent
data_agent_name = "zava-agent"

# (Optional) Name of the workspace if the Data Agent is in a different workspace
workspace_name = None

# (Optional) Name of the output table to store evaluation results (default: "evaluation_output")
# Two tables will be created:
# - "<evaluation_table_name>": contains summary results (e.g., accuracy)
# - "<evaluation_table_name>_steps": contains detailed reasoning and step-by-step execution
evaluation_table_name = "zava_agent_evaluation_output"

# Specify the Data Agent stage: "production" (default) if the Data Agent is published, or "sandbox" otherwise
data_agent_stage = "sandbox"

## 4. Run the baseline evaluation

This is the first full benchmark run using the default evaluator behavior provided by the SDK.

Notice that we pass only the two columns required by the evaluation API: `question` and `expected_answer`.  
The call returns a unique `evaluation_id`, which is later used to inspect row-level details for a specific run.


In [ ]:
from fabric.dataagent.evaluation import evaluate_data_agent

# Run the evaluation and get the evaluation ID
evaluation_id = evaluate_data_agent(
    df[["question", "expected_answer"]],
    data_agent_name,
    workspace_name=workspace_name,
    table_name=evaluation_table_name,
    data_agent_stage=data_agent_stage
)

print(f"Unique ID for the current evaluation run: {evaluation_id}")

## 5. Read the high-level evaluation summary

The summary gives us the first verdict on the benchmark run: how many answers were marked as correct, incorrect, or unclear, together with the corresponding overall percentage.

This is useful as an initial health check, but it is never enough on its own.  
A single aggregate score can hide whether the failures come from the Data Agent, from the evaluator, or from only a few fragile question families.


In [ ]:
from fabric.dataagent.evaluation import get_evaluation_summary

evaluation_table_name = "zava_agent_evaluation_output"

summary_df = get_evaluation_summary(
    table_name=evaluation_table_name,
    verbose=False
)

summary_df

## 6. Inspect the row-level evaluation details

To understand what is really happening, we need the detailed view for a specific `evaluation_id`.

This output lets us inspect the original question, the actual answer returned by the agent, the evaluation verdict, and the associated thread.  
In the article, this step is where the benchmark stops being just a score and becomes diagnostic.


In [ ]:
import pandas as pd
from fabric.dataagent.evaluation import get_evaluation_details

pd.set_option("display.max_colwidth", None)

evaluation_id = "85514c01-a8d4-4046-9a6b-b58a608f5b11"
# "efa339d8-2b25-4017-8d97-da337a5a5b3c"
evaluation_table_name = "zava_agent_evaluation_output"

details_df = get_evaluation_details(
    evaluation_id=evaluation_id,
    table_name=evaluation_table_name,
    get_all_rows=True
)

details_df

## 7. Export the audit table to Excel

Once the detail DataFrame is available, it is often useful to export it to an Excel file for manual review.

This export is especially helpful when we want to inspect borderline cases, compare `actual_answer` against the benchmark ground truth, and build an external audit table for deeper analysis outside the notebook.


In [ ]:
import os
import pandas as pd
from datetime import datetime

folder = "/lakehouse/default/Files/exports"
os.makedirs(folder, exist_ok=True)

filename = f"benchmark_export_{datetime.now():%Y%m%d_%H%M%S}.xlsx"
out_path = os.path.join(folder, filename)

details_df.to_excel(out_path, index=False, sheet_name="Benchmark")

print("Creato:", out_path)
print("Esiste?", os.path.exists(out_path))

## 8. Build a consolidated audit table (query, expected, actual, verdict)

In this step, we transform the raw evaluation export into a cleaner audit artifact that is easier to review and share.

The next cell:
- loads the latest exported evaluation details and the full benchmark file
- merges them on `question` to recover `intent_id`, `question_id`, and `expected_answer`
- fills missing expected answers by propagating a non-null canonical answer from the same `intent_id`
- creates a compact final table with:
    - `question_id`
    - `query`
    - `expected_answer`
    - `actual_answer`
    - `sdk_verdict`
    - `thread_url`
- sorts rows by `question_id` and saves the result to a timestamped Excel file in the Lakehouse `Files` area

This produces a single, analysis-ready audit table that aligns benchmark ground truth with agent outputs and SDK judgments.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# =============================================================================
# INPUT FILES
# =============================================================================
# Export prodotto da get_evaluation_details(..., get_all_rows=False)
export_path = out_path

# Benchmark completo con expected_answer popolata
benchmark_path = "/lakehouse/default/Files/final_benchmark_with_expected_answers.xlsx"

# Output finale
output_xlsx = f"/lakehouse/default/Files/audit_table_with_queries_expected_actual_{datetime.now():%Y%m%d_%H%M%S}.xlsx"


# =============================================================================
# LOAD DATA
# =============================================================================
exp = pd.read_excel(export_path)
bench = pd.read_excel(benchmark_path)

# =============================================================================
# MERGE EXPORT + BENCHMARK
# =============================================================================
# Join sulla question, così recuperiamo intent_id, question_id ed expected_answer
audit = exp.merge(
    bench[["intent_id", "question_id", "question", "expected_answer"]],
    on="question",
    how="left",
    suffixes=("", "_bench")
)

# =============================================================================
# FILL MISSING EXPECTED ANSWERS FROM SAME INTENT
# =============================================================================
# Se per qualche riga il merge non recupera expected_answer,
# proviamo a usare la expected_answer non nulla dello stesso intent_id
canon_map = (
    bench.groupby("intent_id")["expected_answer"]
    .apply(lambda s: next((x for x in s if pd.notna(x)), np.nan))
    .to_dict()
)

audit["expected_answer_final"] = audit["expected_answer"]

mask = audit["expected_answer_final"].isna()
audit.loc[mask, "expected_answer_final"] = audit.loc[mask, "intent_id"].map(canon_map)

# =============================================================================
# BUILD FINAL AUDIT TABLE
# =============================================================================
# Qui assumo che nell'export di evaluation la colonna con il giudizio SDK
# si chiami "evaluation_message" e la risposta dell'agente "actual_answer".
# Se nel tuo file hanno nomi diversi, aggiornali qui sotto.

audit["sdk_verdict"] = audit["evaluation_message"]

final = audit[
    [
        "question_id",
        "question",
        "expected_answer_final",
        "actual_answer",
        "sdk_verdict",
        "thread_url"
    ]
].copy()

final = final.rename(
    columns={
        "question": "query",
        "expected_answer_final": "expected_answer",
    }
)

final = final.sort_values("question_id").reset_index(drop=True)

# =============================================================================
# SAVE OUTPUTS
# =============================================================================
final.to_excel(output_xlsx, index=False)

print(f"Saved Excel: {output_xlsx}")

## 9. Rerun the benchmark with a stricter custom critic prompt

After inspecting the default evaluation behavior, we introduce a stricter replacement prompt designed to reduce evaluator noise.

The goal is not to make the Data Agent smarter, but to make the evaluator behave more consistently by:
- making semantic equivalence explicit
- accepting reasonable numeric rounding
- forcing the critic to return exactly one verdict word with no extra text

This second run lets us test whether part of the weak initial performance came from the evaluation layer rather than from the Data Agent itself.


In [ ]:
from fabric.dataagent.evaluation import evaluate_data_agent

critic_prompt_one_word_strict = """
Given the following query and ground truth, please determine if the most recent answer is equivalent and contains the same required information as the ground truth.

Ignore differences in wording, formatting, Markdown, quotation marks, table versus list style, row order and other non-contradictory details.

If the required entities, metrics and values are the same and semantically equivalent, even with reasonable rounding, respond with 'Yes'. If they differ, respond with 'No'. If it is ambiguous or unclear, respond with 'Unclear'. Only return one word: 'Yes', 'No', or 'Unclear'.

Query: {query}

Ground Truth: {expected_answer}
"""

# Run the evaluation and get the evaluation ID
evaluation_id = evaluate_data_agent(
    df[["question", "expected_answer"]],
    data_agent_name,
    workspace_name=workspace_name,
    table_name=evaluation_table_name,
    data_agent_stage=data_agent_stage,
    critic_prompt=critic_prompt_one_word_strict
)

print(f"Unique ID for the current evaluation run: {evaluation_id}")

## 10. Compare the new summary against the earlier runs

After rerunning the benchmark with the stricter custom critic, we read the summary again.

This is the step where we verify whether the custom prompt actually reduces false negatives and excessive `Unclear` outcomes.  
In the article, this comparison is the key evidence that a substantial share of the initial weakness was due to evaluator instability rather than to the Data Agent alone.


In [ ]:
from fabric.dataagent.evaluation import get_evaluation_summary

evaluation_table_name = "zava_agent_evaluation_output"

summary_df = get_evaluation_summary(
    table_name=evaluation_table_name,
    verbose=False
)

summary_df